In [ ]:
# ======================= CELL 0: Lock to single GPU ===========================
# MUST be first — before any torch/CUDA imports
# bitsandbytes 4-bit doesn't work with DataParallel on 2 GPUs

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("✓ Locked to GPU 0 only")

In [ ]:
# ======================= CELL 1: Install packages =============================

import subprocess
subprocess.run([
    "pip", "install", "-q",
    "transformers", "trl", "peft", "bitsandbytes",
    "accelerate", "datasets", "sacrebleu"
], check=True)

print("✓ All packages installed")

In [ ]:
# ======================= CELL 2: Login & imports ==============================

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

import torch
import json
import random
import sacrebleu
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

random.seed(42)

# Verify GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("✗ NO GPU")

# Find data automatically
DATA_DIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train.jsonl" in files:
        DATA_DIR = root
        break

if DATA_DIR:
    print(f"✓ Data found at: {DATA_DIR}")
else:
    print("✗ train.jsonl not found — add your dataset via + Add Data")

OUTPUT_DIR = "/kaggle/working/luganda_llm_model"
CHECKPOINT_DIR = "/kaggle/working/luganda_checkpoints"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

print("✓ Ready to go")

In [ ]:
# ======================= CELL 3: Load data ====================================

print("Loading datasets...")
dataset = load_dataset("json", data_files={
    "train":      f"{DATA_DIR}/train.jsonl",
    "validation": f"{DATA_DIR}/val.jsonl",
    "test":       f"{DATA_DIR}/test.jsonl",
})

print(f"  Train:      {len(dataset['train'])} examples")
print(f"  Validation: {len(dataset['validation'])} examples")
print(f"  Test:       {len(dataset['test'])} examples")

ex = dataset["train"][0]
print(f"\nSample:")
for msg in ex["messages"]:
    print(f"  {msg['role']}: {msg['content'][:80]}")

In [ ]:
# ======================= CELL 4: Load model in 4-bit ==========================

MODEL_ID = "google/gemma-3-1b-it"
print(f"Loading {MODEL_ID} in 4-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=hf_token,
)

model = prepare_model_for_kbit_training(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model loaded: {total_params / 1e6:.0f}M parameters")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ======================= CELL 5: Attach LoRA adapters =========================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✓ LoRA attached")
print(f"  Trainable: {trainable / 1e6:.2f}M ({100 * trainable / total:.1f}%)")
print(f"  GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ======================= CELL 6: Configure training ===========================

training_config = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    max_length=256,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    seed=42,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

steps_per_epoch = len(dataset["train"]) // (
    training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps
)
total_steps = steps_per_epoch * int(training_config.num_train_epochs)
print(f"✓ Training config ready")
print(f"  Epochs: {training_config.num_train_epochs}")
print(f"  Effective batch size: {training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps}")
print(f"  Estimated steps: {total_steps}")
print(f"  Estimated time: ~{total_steps * 1.5 / 60:.0f} minutes on T4")

In [ ]:
# ======================= CELL 7: Train! =======================================

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_config,
)

print("Starting training...")
print("=" * 50)
train_result = trainer.train()
print("=" * 50)
print(f"✓ Training complete!")
print(f"  Total steps:      {train_result.global_step}")
print(f"  Final train loss: {train_result.training_loss:.4f}")

# Save model
print(f"\nSaving model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Backup zip
import shutil
shutil.make_archive("/kaggle/working/luganda_model", "zip", OUTPUT_DIR)

print(f"✓ Model saved to {OUTPUT_DIR}")
print(f"✓ Backup zip at /kaggle/working/luganda_model.zip")

# List saved files
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"  {f} ({size:.1f} MB)")

In [ ]:
# ======================= CELL 8: Evaluate — compute BLEU ======================

print("Evaluating fine-tuned model...")

en_to_lg_test = []
for ex in dataset["test"]:
    user_msg = ex["messages"][0]["content"].lower()
    is_en_to_lg = any(p in user_msg for p in [
        "translate", "convert this to luganda",
        "english to luganda", "how do you say this in luganda"
    ])
    is_lg_to_en = any(p in user_msg for p in [
        "luganda text to english", "luganda to english",
        "mean in english", "convert this to english"
    ])
    if is_en_to_lg and not is_lg_to_en:
        en_to_lg_test.append(ex)

eval_sample = random.sample(en_to_lg_test, min(100, len(en_to_lg_test)))

predictions = []
references = []

print(f"Running on {len(eval_sample)} examples...")
model.eval()

for i, ex in enumerate(eval_sample):
    prompt = ex["messages"][0]["content"]
    expected = ex["messages"][1]["content"]

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
        )

    prompt_len = inputs["input_ids"].shape[1]
    response = tokenizer.decode(
        output[0][prompt_len:], skip_special_tokens=True
    ).strip()

    predictions.append(response)
    references.append(expected)

    if (i + 1) % 20 == 0:
        print(f"  ... {i + 1}/{len(eval_sample)} done")

bleu = sacrebleu.corpus_bleu(predictions, [references])
chrf = sacrebleu.corpus_chrf(predictions, [references], word_order=2)

print(f"\n{'='*60}")
print(f"RESULTS — FINE-TUNED vs BASELINE vs CRANE")
print(f"{'='*60}")
print(f"{'Model':<35} {'BLEU':>8} {'chrF++':>8}")
print(f"{'-'*55}")
print(f"{'Gemma 3 1B (base, no training)':<35} {'0.06':>8} {'8.84':>8}")
print(f"{'Crane Labs Ganda Gemma 1B':<35} {'6.99':>8} {'---':>8}")
print(f"{'YOUR fine-tuned model':<35} {bleu.score:>8.2f} {chrf.score:>8.2f}")
print(f"{'-'*55}")

if bleu.score > 6.99:
    print(f"\n>>> You BEAT Crane Labs! ({bleu.score:.2f} vs 6.99)")
elif bleu.score > 0.06:
    print(f"\n✓ Significant improvement over baseline ({bleu.score:.2f} vs 0.06)")
    print(f"  Gap to Crane: {6.99 - bleu.score:.2f} BLEU points")

In [ ]:
# ======================= CELL 9: Show examples & save results =================

print(f"\n{'='*60}")
print("SAMPLE TRANSLATIONS")
print(f"{'='*60}")

for i in random.sample(range(len(eval_sample)), min(10, len(eval_sample))):
    lines = eval_sample[i]["messages"][0]["content"].split("\n")
    en_text = lines[-1] if len(lines) > 1 else lines[0]

    print(f"\n  English:    {en_text[:90]}")
    print(f"  Expected:   {references[i][:90]}")
    print(f"  Your model: {predictions[i][:90]}")
    print(f"  {'-'*56}")

results = {
    "bleu": bleu.score,
    "chrf": chrf.score,
    "baseline_bleu": 0.06,
    "crane_bleu": 6.99,
    "eval_size": len(eval_sample),
    "training_loss": train_result.training_loss,
    "examples": [
        {
            "english": eval_sample[i]["messages"][0]["content"],
            "expected": references[i],
            "predicted": predictions[i],
        }
        for i in range(len(eval_sample))
    ],
}

results_path = f"{OUTPUT_DIR}/eval_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n✓ Results saved to: {results_path}")
print(f"\nAll outputs in /kaggle/working/:")
print(f"  luganda_llm_model/   — fine-tuned adapters")
print(f"  luganda_model.zip    — backup for download")
print(f"\nClick 'Save Version' now to permanently save everything.")